# Test-time compute trên GSM8K: CoT, self-consistency, best-of-N

Mục 1 và 2 lý thuyết nói model "nghĩ" bằng token, và có hai cách tiêu thêm compute lúc inference: cho model viết lời
giải dài (chain-of-thought), và sinh nhiều lời giải rồi gộp lại. Notebook này đo cả hai trên 200 câu đầu tập test
GSM8K:

1. `qwen2.5:3b` qua Ollama: trả lời thẳng so với CoT, cùng câu hỏi, có khoảng tin cậy theo cặp.
2. `Qwen2.5-0.5B-Instruct` qua Transformers: greedy so với 16 lời giải lấy mẫu mỗi câu, vẽ maj@k, pass@k và best-of-N
   (verifier là log-prob trung bình của chính model) theo **số token tiêu thêm**.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hằng, hàm. Điền vào chỗ `# TODO`.
- Cell gắn tag `explore` (dòng đầu `# explore: ...`) gọi model, in kết quả; test bỏ qua chúng.
- Phần 1 cần `ollama serve` với `qwen2.5:3b`. Phần 2 chạy bằng kernel `ai-gpu` (GPU 4GB là đủ); kernel CPU chỉ chạy bản
  thu nhỏ để kiểm code và in lại kết quả GPU đã cache nếu có.
- Chấm: `uv run pytest tracks/ai-engineer/modules/15-reasoning-models/exercises/intermediate/01-test-time-compute -q`

In [ ]:
# explore: tìm gốc repo, tải GSM8K nếu chưa có
import os
import sys
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "4")
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))


def dataset_dir(name):
    try:
        from data.download_dataset import ensure
    except ImportError:
        path = ROOT / "data" / "raw" / name
        if not path.is_dir():
            raise RuntimeError(f"Chưa có dữ liệu: uv run python data/download_dataset.py {name}")
        return path
    return ensure(name)


GSM = dataset_dir("gsm8k")
CACHE = GSM / "cache"
CACHE.mkdir(exist_ok=True)
print(GSM, sorted(p.name for p in (GSM / "main").iterdir()))

In [ ]:
import re
from collections.abc import Sequence
from fractions import Fraction

import numpy as np
import pandas as pd

## 1. Dán bộ trích đáp số và bộ bỏ phiếu

Dán từ lời giải của bạn (notebook tự chứa, không import chéo bài):

- bài Basic 01: các regex, `normalize_number`, `_numbers` (hoặc hàm phụ tương đương của bạn), `extract_answer`;
- bài Basic 02: `weighted_vote`, `majority_vote`, `best_of_n`, `pass_at_k`, `maj_at_k`, `coverage_at_k`.

In [ ]:
def normalize_number(text: str) -> str | None:
    """Dán từ bài Basic 01."""
    # TODO
    raise NotImplementedError


def extract_answer(completion: str) -> str | None:
    """Dán từ bài Basic 01."""
    # TODO
    raise NotImplementedError

In [ ]:
def majority_vote(answers: Sequence[str | None]) -> str | None:
    """Dán từ bài Basic 02."""
    # TODO
    raise NotImplementedError


def best_of_n(answers: Sequence[str | None], scores: Sequence[float]) -> str | None:
    """Dán từ bài Basic 02."""
    # TODO
    raise NotImplementedError


def pass_at_k(n: int, c: int, k: int) -> float:
    """Dán từ bài Basic 02."""
    # TODO
    raise NotImplementedError


def maj_at_k(samples: Sequence[Sequence[str | None]], golds: Sequence[str], k: int) -> float:
    """Dán từ bài Basic 02."""
    # TODO
    raise NotImplementedError


def coverage_at_k(samples: Sequence[Sequence[str | None]], golds: Sequence[str], k: int) -> float:
    """Dán từ bài Basic 02."""
    # TODO
    raise NotImplementedError

## 2. Prompt, bảng scaling, bootstrap theo cặp

- `build_messages(question, mode)`: `mode` là `"direct"` hoặc `"cot"`; trả hai message `system` (đúng hằng
  `SYSTEM_DIRECT` hoặc `SYSTEM_COT` bên dưới) và `user` (đúng câu hỏi, không thêm chữ). Mode khác thì `ValueError`.
  `SYSTEM_COT` là prompt toán mà Qwen khuyến nghị cho dòng Qwen2.5.
- `scaling_table(answers, tokens, logprobs, golds, ks) -> DataFrame` cột đúng thứ tự `k, maj, pass, bon_logprob,
  tokens`, một dòng mỗi `k`. `answers[i]`, `tokens[i]`, `logprobs[i]` là đáp số, số token sinh ra và log-prob trung bình
  mỗi token của từng lời giải lấy mẫu của câu `i`.
  - `maj`, `pass`: `maj_at_k`, `coverage_at_k`.
  - `bon_logprob`: như `maj_at_k` (khối `k` liên tiếp, bỏ phần dư, trung bình khối rồi trung bình câu) nhưng mỗi khối
    chọn bằng `best_of_n` với điểm là log-prob trung bình.
  - `tokens`: tổng token sinh ra của một khối, trung bình theo khối rồi theo câu: chi phí của "chạy k lần".
- `paired_bootstrap(correct_a, correct_b, n_boot=2000, seed=0) -> (diff, lo, hi)`: `diff` là accuracy của B trừ của A;
  lấy mẫu lại `n` câu có hoàn lại `n_boot` lần, **cùng chỉ số cho cả hai cột**, `lo`/`hi` là phân vị 2.5 và 97.5 của
  hiệu (module 11 mục 7). Dùng `np.random.default_rng(seed)`; trả ba `float`.

In [ ]:
SYSTEM_DIRECT = "Answer with only the final number. Do not explain."
SYSTEM_COT = "Please reason step by step, and put your final answer within \\boxed{}."


def build_messages(question: str, mode: str) -> list[dict]:
    """[system, user] cho mode "direct" hoặc "cot"; mode khác: ValueError."""
    # TODO
    raise NotImplementedError


def scaling_table(answers: list[list[str | None]], tokens: list[list[int]], logprobs: list[list[float]],
                  golds: list[str], ks: list[int]) -> pd.DataFrame:
    """Một dòng mỗi k: k, maj, pass, bon_logprob, tokens."""
    # TODO
    raise NotImplementedError


def paired_bootstrap(correct_a, correct_b, n_boot: int = 2000, seed: int = 0) -> tuple[float, float, float]:
    """(accuracy B - accuracy A, phân vị 2.5, phân vị 97.5) qua bootstrap theo cặp."""
    # TODO
    raise NotImplementedError

## 3. Trả lời thẳng hay CoT: `qwen2.5:3b` qua Ollama

Gọi thẳng REST của Ollama (module 09 mục 6) để đặt `seed`, `num_predict` và đọc số token sinh ra (`eval_count`).
`temperature` 0, `seed` 0. Kết quả cache theo `(mode, chỉ số câu)` trong `data/raw/gsm8k/cache/`, chạy lại không gọi
lại. Khoảng 400 lượt gọi: chừng 20 phút trên GPU của PC (CoT chiếm gần hết), lâu hơn nhiều nếu Ollama chạy trên CPU.

In [ ]:
# explore: 200 câu ghim, gọi Ollama hai chế độ
import json
import time

import httpx

test = pd.read_parquet(GSM / "main" / "test-00000-of-00001.parquet")
N_PINNED = 200
pinned = test.iloc[:N_PINNED]
golds = [normalize_number(a.split("####")[-1]) for a in pinned.answer]
OLLAMA = os.environ.get("OLLAMA_HOST", "http://localhost:11434")
NUM_PREDICT = {"direct": 32, "cot": 768}
cache_file = CACHE / "m15-ttc-ollama.json"
cache = json.loads(cache_file.read_text(encoding="utf-8")) if cache_file.exists() else {}


def call(mode, i, question):
    key = f"{mode}|{i}"
    if key not in cache:
        payload = {"model": "qwen2.5:3b", "messages": build_messages(question, mode), "stream": False,
                   "options": {"temperature": 0, "seed": 0, "num_predict": NUM_PREDICT[mode]}}
        r = httpx.post(f"{OLLAMA}/api/chat", json=payload, timeout=600).json()
        cache[key] = {"reply": r["message"]["content"], "tokens": r.get("eval_count", 0),
                      "seconds": (r.get("total_duration", 0) - r.get("load_duration", 0)) / 1e9,
                      "truncated": r.get("done_reason") == "length"}
    return cache[key]


ollama_rows = {}
for mode in ("direct", "cot"):
    t0 = time.perf_counter()
    ollama_rows[mode] = [call(mode, i, q) for i, q in enumerate(pinned.question)]
    cache_file.write_text(json.dumps(cache, ensure_ascii=False), encoding="utf-8")
    print(mode, f"{time.perf_counter() - t0:.0f} s (0 s nếu đã cache)")

In [ ]:
# explore: bảng direct so với CoT, hiệu accuracy kèm khoảng tin cậy 95% theo cặp
correct = {m: np.array([extract_answer(r["reply"]) == g for r, g in zip(rows, golds)]) for m, rows in ollama_rows.items()}
summary = pd.DataFrame({m: {"accuracy": correct[m].mean(),
                            "token ra TB": np.mean([r["tokens"] for r in rows]),
                            "giây trung vị": np.median([r["seconds"] for r in rows]),
                            "bị cắt (num_predict)": sum(r["truncated"] for r in rows)}
                        for m, rows in ollama_rows.items()}).T
display(summary.round(3))
diff, lo, hi = paired_bootstrap(correct["direct"], correct["cot"])
print(f"CoT - direct: {diff:+.3f}  [{lo:+.3f}, {hi:+.3f}]")
print("câu direct đúng mà CoT sai:", int((correct["direct"] & ~correct["cot"]).sum()))
for i in np.where(correct["direct"] & ~correct["cot"])[0][:2]:
    print(f"\n--- câu {i}, đáp án {golds[i]}\n", ollama_rows["cot"][i]["reply"][-500:])

## 4. Nhiều lời giải: `Qwen2.5-0.5B-Instruct`

Cùng 200 câu, prompt CoT. Một lời giải greedy và 16 lời giải lấy mẫu (`temperature` 0.7) mỗi câu, tối đa 512 token.
Log-prob trung bình mỗi token của từng lời giải tính lại bằng một lượt forward (model ở `temperature` 1), làm điểm cho
best-of-N. Khoảng 22 phút cho 3 400 lời giải trên RTX 3060 của PC (GPU 4GB chậm hơn); lời giải cache ra đĩa trước bước
log-prob. Kernel CPU chạy 8 câu, 4 lời giải mỗi câu.

In [ ]:
# explore: nạp model, sinh greedy và lấy mẫu, cache ra đĩa
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

GPU = torch.cuda.is_available()
DEVICE = "cuda" if GPU else "cpu"
MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
N_Q, K, MAX_NEW, BATCH = (200, 16, 512, 64) if GPU else (8, 4, 256, 16)
tok = AutoTokenizer.from_pretrained(MODEL)
tok.padding_side = "left"
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.bfloat16 if GPU else torch.float32).to(DEVICE).eval()
STOP = {tok.eos_token_id, tok.pad_token_id, tok.convert_tokens_to_ids("<|im_end|>")}


@torch.no_grad()
def sample(questions, n, temperature, seed=0):
    prompts = [tok.apply_chat_template(build_messages(q, "cot"), tokenize=False, add_generation_prompt=True)
               for q in questions for _ in range(n)]
    out = []
    for s in range(0, len(prompts), BATCH):
        torch.manual_seed(seed + s)
        enc = tok(prompts[s:s + BATCH], return_tensors="pt", padding=True, add_special_tokens=False).to(DEVICE)
        kw = (dict(do_sample=True, temperature=temperature, top_p=1.0, top_k=0) if temperature > 0
              else dict(do_sample=False, temperature=None, top_p=None, top_k=None))
        gen = model.generate(**enc, max_new_tokens=MAX_NEW, pad_token_id=tok.pad_token_id, **kw)
        width = enc["input_ids"].shape[1]
        for row in range(gen.shape[0]):
            ids = gen[row, width:].tolist()
            end = next((j + 1 for j, t in enumerate(ids) if t in STOP), len(ids))
            out.append({"prompt_ids": enc["input_ids"][row][enc["attention_mask"][row].bool()].tolist(),
                        "ids": ids[:end], "text": tok.decode(ids[:end], skip_special_tokens=True)})
    return out


@torch.no_grad()
def mean_logprob(items, batch=4):
    scores = []
    for s in range(0, len(items), batch):
        chunk = items[s:s + batch]
        seqs = [it["prompt_ids"] + it["ids"] for it in chunk]
        width = max(map(len, seqs))
        ids = torch.full((len(seqs), width), tok.pad_token_id)
        mask = torch.zeros((len(seqs), width), dtype=torch.long)
        for r, seq in enumerate(seqs):
            ids[r, width - len(seq):] = torch.tensor(seq)
            mask[r, width - len(seq):] = 1
        keep = max(len(it["ids"]) for it in chunk)
        logits = model(input_ids=ids.to(DEVICE), attention_mask=mask.to(DEVICE), logits_to_keep=keep + 1).logits[:, :-1]
        for r, it in enumerate(chunk):
            n = len(it["ids"])
            row = logits[r, keep - n:].float()
            target = torch.tensor(it["ids"], device=DEVICE)
            scores.append(float((row.gather(-1, target[:, None]).squeeze(-1) - torch.logsumexp(row, -1)).mean()))
    return scores


questions, q_golds = list(pinned.question[:N_Q]), golds[:N_Q]
hf_file = CACHE / ("m15-ttc-hf.json" if GPU else "m15-ttc-hf-cpu.json")
raw_file = CACHE / ("m15-ttc-hf-raw.json" if GPU else "m15-ttc-hf-raw-cpu.json")
if hf_file.exists():
    hf = json.loads(hf_file.read_text(encoding="utf-8"))
else:
    if raw_file.exists():
        raw = json.loads(raw_file.read_text(encoding="utf-8"))
    else:
        t0 = time.perf_counter()
        greedy = sample(questions, 1, 0.0)
        t_greedy = time.perf_counter() - t0
        sampled = sample(questions, K, 0.7)
        raw = {"greedy": greedy, "sampled": sampled,
               "seconds": {"greedy": t_greedy, "sampled": time.perf_counter() - t0 - t_greedy}}
        raw_file.write_text(json.dumps(raw), encoding="utf-8")
    t0 = time.perf_counter()
    lps = mean_logprob(raw["sampled"])
    hf = {"greedy": [{"answer": extract_answer(g["text"]), "tokens": len(g["ids"])} for g in raw["greedy"]],
          "samples": [{"answer": extract_answer(s["text"]), "tokens": len(s["ids"]), "logprob": lp}
                      for s, lp in zip(raw["sampled"], lps)],
          "examples": [s["text"] for s in raw["sampled"][:K]],
          "seconds": {**raw["seconds"], "logprob": time.perf_counter() - t0},
          "device": torch.cuda.get_device_name() if GPU else "cpu"}
    hf_file.write_text(json.dumps(hf, ensure_ascii=False), encoding="utf-8")
print(hf["device"], {k: round(v) for k, v in hf["seconds"].items()})

In [ ]:
# explore: greedy so với một mẫu, maj@k, pass@k, best-of-N theo log-prob; vẽ theo token
import matplotlib.pyplot as plt


def curves(hf, golds, K):
    n = len(golds)
    S = hf["samples"]
    answers = [[S[i * K + j]["answer"] for j in range(K)] for i in range(n)]
    tokens = [[S[i * K + j]["tokens"] for j in range(K)] for i in range(n)]
    lps = [[S[i * K + j]["logprob"] for j in range(K)] for i in range(n)]
    ks = [k for k in (1, 2, 4, 8, 16) if k <= K]
    table = scaling_table(answers, tokens, lps, golds, ks)
    greedy_ok = np.array([g["answer"] == gold for g, gold in zip(hf["greedy"], golds)])
    return table, greedy_ok, float(np.mean([g["tokens"] for g in hf["greedy"]])), answers


def show(hf, golds, K, title):
    table, greedy_ok, greedy_tokens, answers = curves(hf, golds, K)
    print(f"{title}: greedy accuracy {greedy_ok.mean():.3f} ({greedy_tokens:.0f} token TB)")
    display(table.round(3))
    maj_k = np.array([majority_vote(a) == g for a, g in zip(answers, golds)])
    d, lo, hi = paired_bootstrap(greedy_ok, maj_k)
    print(f"maj@{K} - greedy: {d:+.3f} [{lo:+.3f}, {hi:+.3f}]")
    fig, ax = plt.subplots(figsize=(6, 4))
    for col, label in (("pass", "pass@k (verifier hoàn hảo)"), ("maj", "maj@k (bỏ phiếu)"),
                       ("bon_logprob", "best-of-k theo log-prob")):
        ax.plot(table["tokens"], table[col], marker="o", label=label)
    ax.axhline(greedy_ok.mean(), color="gray", ls="--", label="greedy")
    ax.set_xscale("log", base=2)
    ax.set_xlabel("token sinh ra mỗi câu (log2)")
    ax.set_ylabel("accuracy")
    ax.set_title(title)
    ax.legend()
    plt.show()
    return table


show(hf, q_golds, K, f"Qwen2.5-0.5B-Instruct, {len(q_golds)} câu ({hf['device']})")
gpu_file = CACHE / "m15-ttc-hf.json"
if not GPU and gpu_file.exists():
    show(json.loads(gpu_file.read_text(encoding="utf-8")), golds, 16, "Kết quả GPU đã cache, 200 câu")

## Đọc kết quả

- CoT hơn trả lời thẳng bao nhiêu với `qwen2.5:3b`, khoảng tin cậy có chứa 0 không? Đổi lại tốn gấp bao nhiêu lần token
  ra và độ trễ? Có câu nào trả lời thẳng đúng mà CoT sai không, đọc lời giải CoT của nó.
- Với 0.5B: một mẫu ở `temperature` 0.7 (`maj` ở k = 1) so với greedy thế nào? Self-consistency cần bao nhiêu mẫu mới
  vượt greedy, và khi đó tốn gấp bao nhiêu token?
- Khoảng cách giữa pass@16 và maj@16 nói gì về giá trị của một verifier tốt? Best-of-N theo log-prob của chính model đứng
  ở đâu giữa hai đường: log-prob có phải một verifier tốt không?
- Đường maj@k có phẳng dần không? Ước lượng: muốn thêm vài điểm accuracy nữa thì tốn thêm bao nhiêu token, và so với
  việc dùng model 3B (phần 3) thì cái nào rẻ hơn?

Sau đó làm phần trắc nghiệm của bài trong app.